# Pattern 4: Evaluator-optimizer

**Idea:** one call **generates** a draft, a second call **evaluates** it against explicit criteria and returns feedback, and the generator **revises** using that feedback. Repeat until the evaluator passes it or you hit a maximum number of rounds.

```
        ┌──────────────── feedback ────────────────┐
        ▼                                          │
task ─▶ [LLM: generator] ─▶ draft ─▶ [LLM: evaluator] ─▶ PASS? ─▶ yes ─▶ output
                                                   │
                                          no, and round < MAX
```

**Why it exists:** first drafts are rarely the best. When you can state clear quality criteria, a separate critic step reliably improves the result, just like code review improves code.

**Two things that make it work:**
1. **Explicit criteria.** "Is it good?" gives vague feedback. "Is it under 30 words, does it mention X, does it avoid Y?" gives actionable feedback.
2. **A hard stop.** `MAX_ROUNDS` in code. Without it, a picky evaluator can loop forever and burn tokens.

**Trade-off:** each round is at least two calls. Only worth it when quality really matters and criteria are clear.

**Java analogy:** a CI pipeline with a red/green loop: build, run tests, read the failures, fix, rebuild. The evaluator is the test suite; `MAX_ROUNDS` is the pipeline timeout.

## Step 0: Setup (same in every notebook)

Every notebook starts with the same three setup cells, so you can open any of them alone without running the others first.

1. **Load configuration** from the `.env` file in the repo root (three variables: `PORTKEY_BASE_URL`, `PORTKEY_API_KEY`, `MODEL`).
2. **Create the client.** We use the official `anthropic` Python SDK and point it at the Portkey gateway instead of `api.anthropic.com`.
3. **Define a helper**, `call_claude()`, that wraps one API call and prints everything: the prompt going in, the text coming out, token usage, and timing.

> **Java analogy:** `.env` + `python-dotenv` is like `application.properties` + Spring's `@Value`. The client is a `RestTemplate`/`WebClient` bean configured with a base URL and a default header. Portkey sits in between like an API gateway (think Spring Cloud Gateway): it holds the real Anthropic credentials, routes the call, and logs it.

In [ ]:
# --- Cell 0.1: load configuration from .env ---------------------------------
import os
import json
import time
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) walks UP from the notebook's folder until it finds a .env file.
# That's why the .env can live in the repo root while notebooks live in notebooks/.
env_path = find_dotenv(usecwd=True)
print(f"[setup] .env file found at: {env_path or 'NOT FOUND (using existing environment variables)'}")
load_dotenv(env_path, override=False)  # override=False: real env vars win over the file

PORTKEY_BASE_URL = os.getenv("PORTKEY_BASE_URL", "https://api.portkey.ai")
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
MODEL = os.getenv("MODEL")

# Fail fast with a clear message instead of a confusing HTTP 401 later.
assert PORTKEY_API_KEY, "PORTKEY_API_KEY is missing. Copy .env.example to .env and fill it in."
assert MODEL, "MODEL is missing. Example: @anthropic/claude-sonnet-4-5-20250929"

# Never print a full secret. Show just enough to confirm which key is loaded.
masked_key = PORTKEY_API_KEY[:4] + "..." + PORTKEY_API_KEY[-4:]
print(f"[setup] PORTKEY_BASE_URL = {PORTKEY_BASE_URL}")
print(f"[setup] PORTKEY_API_KEY  = {masked_key}")
print(f"[setup] MODEL            = {MODEL}")

In [ ]:
# --- Cell 0.2: create the Anthropic client, routed through Portkey -----------
import anthropic

client = anthropic.Anthropic(
    # The SDK requires an api_key, but Portkey holds the real Anthropic key for us.
    # Portkey identifies us through the x-portkey-api-key header instead.
    api_key="not-used-portkey-handles-auth",
    base_url=PORTKEY_BASE_URL,
    default_headers={"x-portkey-api-key": PORTKEY_API_KEY},
)
print(f"[setup] Anthropic SDK version: {anthropic.__version__}")
print(f"[setup] Client ready. Every request goes to {PORTKEY_BASE_URL}/v1/messages")

In [ ]:
# --- Cell 0.3: a logging helper around ONE model call -------------------------
# Everything in these notebooks is built from this single building block.
# The patterns are not new API features: they are just different ways of
# arranging plain calls like this one with ordinary Python code.

def call_claude(prompt, system=None, label="call", max_tokens=1024, verbose=True):
    """Send one user message to Claude and return the response text.

    prompt     -> the user message (str)
    system     -> optional system prompt: the model's role and rules
    label      -> a name for this call so the logs are easy to follow
    verbose    -> print the full prompt and response (set False inside threads)
    """
    if verbose:
        print("=" * 78)
        print(f"[{label}] >>> REQUEST")
        if system:
            print(f"[{label}] system: {system}")
        print(f"[{label}] user:\n{prompt}")

    started = time.perf_counter()
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    elapsed = time.perf_counter() - started

    # response.content is a LIST of blocks. For plain text calls there is one text block.
    text = "".join(block.text for block in response.content if block.type == "text")

    if verbose:
        print(f"[{label}] <<< RESPONSE ({elapsed:.2f}s, "
              f"in={response.usage.input_tokens} tok, out={response.usage.output_tokens} tok, "
              f"stop_reason={response.stop_reason})")
        print(text)
        print("=" * 78)
    if response.stop_reason == "max_tokens":
        print(f"[{label}] WARNING: response was cut off by max_tokens")
    return text


def parse_json(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

print("[setup] call_claude() and parse_json() are defined")

In [ ]:
# --- Cell 0.4: smoke test. If this works, your .env and gateway are fine -------
call_claude("Reply with exactly: gateway OK", label="smoke-test", max_tokens=20)

## Step 1: The task and the evaluation criteria

The criteria are written down **once** and given to the evaluator. Write them like test assertions.

In [ ]:
task = "Write a one-sentence product tagline for a CLI tool that syncs Windows environment variables from a git-tracked file."

criteria = """
1. Exactly one sentence.
2. Under 15 words.
3. Mentions BOTH git and environment variables (or 'env vars').
4. No marketing cliches like 'revolutionize', 'seamless', 'game-changer'.
"""
MAX_ROUNDS = 3
print(f"[task] {task}")
print(f"[criteria]{criteria}")
print(f"[config] MAX_ROUNDS = {MAX_ROUNDS}")

## Step 2: Define the two roles

- `generate()` writes a draft. On later rounds it also receives the previous draft and the feedback.
- `evaluate()` returns JSON with `pass` (true/false) and `feedback`, so our loop can branch on it with code.

In [ ]:
generator_system = "You write concise product copy. Output only the tagline, nothing else."

evaluator_system = (
    "You are a strict evaluator. Check the draft against EACH criterion. "
    "Respond ONLY with JSON: {\"pass\": true|false, \"feedback\": \"which criteria failed and how to fix them\"}. "
    "pass is true only if ALL criteria are met."
)

def generate(task, previous_draft=None, feedback=None, round_no=1):
    prompt = f"Task: {task}"
    if previous_draft:
        # This is the 'optimizer' part: the generator sees its own last attempt plus the critique.
        prompt += f"\n\nPrevious draft: {previous_draft}\nPrevious feedback: {feedback}\nWrite an improved version."
    return call_claude(prompt, system=generator_system, label=f"round{round_no}-generator", max_tokens=100).strip()

def evaluate(draft, round_no=1):
    prompt = f"Criteria:{criteria}\nDraft: {draft}"
    raw = call_claude(prompt, system=evaluator_system, label=f"round{round_no}-evaluator", max_tokens=300)
    return parse_json(raw)

print("[setup] generate() and evaluate() defined")

## Step 3: The loop

Read the log for each round: draft, verdict, feedback. You should see the draft change in response to the specific feedback.

In [ ]:
draft, feedback = None, None
history = []

for round_no in range(1, MAX_ROUNDS + 1):
    print(f"\n{'#' * 30} ROUND {round_no} of {MAX_ROUNDS} {'#' * 30}")
    draft = generate(task, previous_draft=draft, feedback=feedback, round_no=round_no)
    verdict = evaluate(draft, round_no=round_no)
    history.append({"round": round_no, "draft": draft, **verdict})

    print(f"[round {round_no}] draft    = {draft}")
    print(f"[round {round_no}] pass     = {verdict['pass']}")
    print(f"[round {round_no}] feedback = {verdict['feedback']}")

    if verdict["pass"]:
        print(f"\n[loop] PASSED in round {round_no}. Stopping.")
        break
    feedback = verdict["feedback"]
else:
    # for/else: this runs only if the loop finished WITHOUT break
    print(f"\n[loop] Hit MAX_ROUNDS={MAX_ROUNDS} without passing. Returning the last draft anyway.")

print(f"\n[final] {draft}")

## Step 4: Review the history

Keeping a history is cheap and invaluable for debugging: you can see exactly which criterion kept failing.

In [ ]:
for h in history:
    status = "PASS" if h["pass"] else "FAIL"
    print(f"round {h['round']}: [{status}] {h['draft']}")
    if not h["pass"]:
        print(f"           feedback: {h['feedback']}")

## Step 5: Add a deterministic check next to the LLM evaluator

Some criteria don't need a model at all. Word count is just `len(text.split())`. Code checks are free, instant, and never wrong. Use the LLM evaluator for criteria that need judgment (tone, cliches) and code for criteria that are mechanical.

In [ ]:
word_count = len(draft.split())
print(f"[code-check] word count = {word_count} -> {'OK' if word_count < 15 else 'TOO LONG'}")
print(f"[code-check] mentions git = {'git' in draft.lower()}")

## Recap

| Piece | Role | Implemented by |
|---|---|---|
| Generator | write / rewrite | model call |
| Evaluator | judge against criteria | model call (+ code checks) |
| Loop and stop condition | decide whether to continue | **your Python code** |

## Experiments to try

1. Make the criteria stricter (for example, "under 8 words") and watch it need more rounds, or hit `MAX_ROUNDS`.
2. Replace the criteria with just "Is it good?" and compare the quality of the feedback.
3. Move criterion 2 (word count) out of the LLM evaluator and into code, then combine both verdicts.